# 01 — Zero-Leakage Pakistani PK-VTON Dataset Builder + Roboflow 100-Outfit Export
- **Strict Deduplication:** Excludes all `.source.*` twin files + SHA-256 + perceptual dHash.
- **Zero-Leakage SKU Split:** Splits by outfit SKU (`85%` Train SKUs / `15%` Completely Unseen Test SKUs).
- **Roboflow Export:** Automatically saves `roboflow_100_unique_for_dupatta.zip` (100 unique outfits, 1 per SKU) for 1-click SAM dupatta annotation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers torch pillow numpy pandas tqdm opencv-python-headless


In [ ]:
import hashlib
import shutil
import zipfile
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# Isolated FYP-2 Output Workspace
RESEARCH_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_FYP2_Research')
DATA_DIR = RESEARCH_ROOT / '01_PK_VTON_DATASET'
PAIRED_TRAIN = DATA_DIR / 'paired_train'
PAIRED_TEST = DATA_DIR / 'paired_test'
GOLD_DIR = DATA_DIR / 'gold_4class_masks'
GOLD_DIR.mkdir(parents=True, exist_ok=True)
(DATA_DIR / 'dupatta_4class_masks').mkdir(parents=True, exist_ok=True)

# Read-Only FYP-1 Pakistani Sources (NEVER modified)
FYP1_ROOT = Path('/content/drive/MyDrive/FashionFit_PK_Research_V3_2026-11-17')
PK_CATALOG = FYP1_ROOT / '02_DATA' / 'intake_2026-11-17' / 'data' / 'new_catalog_candidates'
PK_SEG_DIR = FYP1_ROOT / '02_DATA' / 'intake_2026-11-17' / 'data' / 'segmentation'
FYP1_B2_CKPT = FYP1_ROOT / '10_RUNS' / '05_b2_r2_decoder_plain' / 'best_checkpoint'

print('PK Catalog exists:', PK_CATALOG.exists())
print('PK Seg dir exists:', PK_SEG_DIR.exists())
print('FYP-1 Adapted R2 SegFormer-B2 exists:', FYP1_B2_CKPT.exists())


In [ ]:
# 1. Strict Quality Filtering, .source.* Removal & Perceptual dHash Deduplication
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
records = []
seen_sha = set()
seen_dhash = set()
rejected = {'small': 0, 'aspect': 0, 'blurry': 0, 'source_twin': 0, 'dup_hash': 0, 'excluded': 0}

def compute_dhash(im_rgb: Image.Image) -> str:
    gray = np.array(im_rgb.convert('L').resize((9, 8), Image.BILINEAR))
    diff = gray[:, 1:] > gray[:, :-1]
    return ''.join('1' if b else '0' for b in diff.flatten())

for src_name, src_folder in [('pk_catalog', PK_CATALOG), ('pk_segmentation', PK_SEG_DIR)]:
    if not src_folder.exists():
        continue
    for p in tqdm(sorted(src_folder.rglob('*')), desc=f'Auditing {src_name}'):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        name_low = p.name.lower()
        low = str(p).lower()
        # CRITICAL: Skip .source.jpg / .source.png twin files to prevent train/test leakage!
        if '.source.' in name_low:
            rejected['source_twin'] += 1
            continue
        # Strictly exclude synthetic avatars, masks, overlays, and blind triptychs
        if any(tok in low for tok in ['demo_only', 'a001', 'a002', 'a003', 'a004', 'a005', 'a006', 'mask', 'overlay', 'blind']):
            rejected['excluded'] += 1
            continue
        try:
            raw = p.read_bytes()
            h = hashlib.sha256(raw).hexdigest()[:16]
            if h in seen_sha:
                rejected['dup_hash'] += 1
                continue
            with Image.open(p) as im:
                im = im.convert('RGB')
                w, h_px = im.size
                if h_px < 512 or w < 360:
                    rejected['small'] += 1
                    continue
                if h_px <= w:  # Vertical full-body portraits only
                    rejected['aspect'] += 1
                    continue
                dh = compute_dhash(im)
                if dh in seen_dhash:
                    rejected['dup_hash'] += 1
                    continue
                gray = cv2.cvtColor(np.array(im), cv2.COLOR_RGB2GRAY)
                sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
                if sharpness < 25.0:
                    rejected['blurry'] += 1
                    continue
            seen_sha.add(h)
            seen_dhash.add(dh)
            sku = p.parent.name if src_name == 'pk_catalog' else f'pilot_{p.stem}'
            brand = p.parent.parent.name if src_name == 'pk_catalog' else 'reviewed_pilot'
            records.append({
                'image_id': f'pk_{h}',
                'sku': sku,
                'brand': brand,
                'source': src_name,
                'path': str(p),
                'width': w,
                'height': h_px,
                'sharpness': round(sharpness, 1),
            })
        except Exception:
            continue

df = pd.DataFrame(records)
manifest_path = DATA_DIR / 'pk_vton_manifest.csv'
df.to_csv(manifest_path, index=False)
print(f'100% UNIQUE PAKISTANI IMAGES: {len(df)} across {df["sku"].nunique()} unique outfits/SKUs')
print('Brands:', df['brand'].value_counts().to_dict())
print('Rejected stats:', rejected)

# Export 100 unique outfits (1 image per SKU, balanced across brands) as a ZIP for Roboflow 1-click SAM annotation
unique_sku_df = df.drop_duplicates(subset=['sku']).copy()
# Prioritize Generation (highest dupatta ratio) + Ethnc + Limelight + Reviewed Pilot
roboflow_picks = []
for b, quota in [('generation', 40), ('ethnc', 25), ('limelight', 25), ('reviewed_pilot', 15)]:
    sub = unique_sku_df[unique_sku_df['brand'] == b]
    if len(sub) > 0:
        roboflow_picks.append(sub.sample(n=min(quota, len(sub)), random_state=42))
rf_df = pd.concat(roboflow_picks).head(100).reset_index(drop=True)

zip_path = DATA_DIR / 'roboflow_100_unique_for_dupatta.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for _, row in rf_df.iterrows():
        src_p = Path(row['path'])
        zf.write(src_p, arcname=f"{row['image_id']}_{row['sku']}.jpg")
rf_df.to_csv(DATA_DIR / 'roboflow_100_manifest.csv', index=False)
print(f'EXPORTED {len(rf_df)} UNIQUE OUTFITS FOR ROBOFLOW -> {zip_path}')


In [ ]:
# 2. Rebuild ZERO-LEAKAGE SKU-Grouped Paired Triples (Train SKUs vs Unseen Test SKUs)
# Clean old folders so zero stale .source.* twins remain
for split_dir in [PAIRED_TRAIN, PAIRED_TEST]:
    if split_dir.exists():
        shutil.rmtree(split_dir)
    for sub in ['person', 'garment', 'mask']:
        (split_dir / sub).mkdir(parents=True, exist_ok=True)

ckpt_source = str(FYP1_B2_CKPT) if FYP1_B2_CKPT.exists() else 'mattmdjaga/segformer_b2_clothes'
print('Loading SegFormer-B2 parser from:', ckpt_source)
seg_proc = SegformerImageProcessor.from_pretrained(ckpt_source)
seg_model = SegformerForSemanticSegmentation.from_pretrained(ckpt_source).to(device).eval()
is_adapted_4class = seg_model.config.num_labels == 4

@torch.no_grad()
def extract_kurta_mask(im: Image.Image) -> np.ndarray:
    w, h = im.size
    enc = seg_proc(images=im, return_tensors='pt').to(device)
    logits = seg_model(**enc).logits
    up = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
    pred = up.argmax(dim=1)[0].cpu().numpy()
    if is_adapted_4class:
        return (pred == 1).astype(np.uint8)
    return np.isin(pred, [4, 7]).astype(np.uint8)

# Strictly group by SKU so NO outfit ever appears in both train and test!
rng = np.random.default_rng(42)
unique_skus = np.array(sorted(df['sku'].unique()))
rng.shuffle(unique_skus)
n_test_skus = max(25, int(0.15 * len(unique_skus)))
test_skus = set(unique_skus[:n_test_skus])

kept_train, kept_test = 0, 0
for _, row in tqdm(df.iterrows(), total=len(df), desc='Building zero-leakage paired triples'):
    im = Image.open(row['path']).convert('RGB')
    k_mask = extract_kurta_mask(im)
    cov = float(k_mask.mean())
    if cov < 0.10 or cov > 0.55:
        continue

    is_test = row['sku'] in test_skus
    split_dir = PAIRED_TEST if is_test else PAIRED_TRAIN
    stem = row['image_id']

    im.save(split_dir / 'person' / f'{stem}.jpg', quality=95)
    Image.fromarray(k_mask * 255).save(split_dir / 'mask' / f'{stem}.png')
    arr = np.array(im)
    cutout = np.full_like(arr, 255)
    cutout[k_mask == 1] = arr[k_mask == 1]
    Image.fromarray(cutout).save(split_dir / 'garment' / f'{stem}.png')

    if is_test:
        kept_test += 1
    else:
        kept_train += 1

print(f'DONE! Zero-Leakage Paired Triples -> Train: {kept_train} | Unseen-SKU Test: {kept_test}')
print(f'Train SKUs: {len(unique_skus) - n_test_skus} | Unseen Test SKUs: {n_test_skus} (0.0% overlap!)')
